# Samle data

In [1]:
import sys
import torch
import shutil
import itertools
from torch.utils.data import ConcatDataset, DataLoader



import numpy as np

from pathlib import Path

ROOT_DIR = Path().resolve().parents[1]
sys.path.insert(0, str(ROOT_DIR))

from hesel_scraper.bout_dump import BOUTHESELInfo
from hesel_scraper.bout_phys import BOUTHESELPhys
from SciML.PINO_z.utils.data_loader import make_dataloaders

root1 = ROOT_DIR / r"sim_data/Alexander_e"
root2 = ROOT_DIR / r"sim_data/Alexander_phi_0"

info1 = BOUTHESELInfo(root1)
info2 = BOUTHESELInfo(root2)


In [ ]:
# Til endelige test
train_config = {
    "resume": False,
    "root" : ROOT_DIR / r"sim_data/Alexander_std_256_1",
    "data_dir": ROOT_DIR / r"Experimenter/PINO/test_run",
    "seed": np.random.randint(0, 2**32 - 1),
    "status_frequency": 1,
    "flush_frequency": 50,
    "z_width": 3,
    "num_eq_chunk": 18,
    "batch_size": 130,
    "train_split": 0.8,
    "val_split": 0.1,
    "shuffle": True, # <- Hvis False køres datasæt 1 først, derefter datasæt 2. Hvis True, blandes de to datasæt.
    "num_workers": 4,
    "prefetch_factor": 1,
    "pin_memory": True,
    "lr": 1e-7, # <- Min lr, max er sat til 5e-4 i CyclicLR
    "epochs": 10,
    "early_stopping_patience": 2,
    "early_stopping_min_delta": 0.0,
    "fno": {
        "n_modes": (150, 150),
        "in_channels": 12,
        "out_channels": 4,
        "hidden_channels": 30,
        "positional_embedding": None,
    }
}

In [ ]:
data_loader_setup1 = {
    "TSSplit": True,
    "train_split": 0.8,
    "val_split": 0.1,
}

data_loader_setup2 = {
    "TSSplit": False,
    "train_split": 0.8,
    "val_split": 0.1,
}


data_loader_dict = {
    "z_width": train_config["z_width"],
    "batch_size": train_config["batch_size"],
    "val_split": train_config["val_split"],
    "shuffle": train_config["shuffle"],
    "num_workers": train_config["num_workers"],
    "prefetch_factor": train_config["prefetch_factor"],
    "pin_memory": train_config["pin_memory"],
}

for key, value in data_loader_dict.items():
    data_loader_setup1[key] = value
    data_loader_setup2[key] = value


train_loader1, val_loader1, test_loader1 = make_dataloaders(info1, data_loader_setup1)
train_loader2, val_loader2, test_loader2 = make_dataloaders(info2, data_loader_setup2)


# combine the two train datasets so each batch can contain samples from both info1 and info2
train_dataset = ConcatDataset([train_loader1.dataset, train_loader2.dataset])
train_loader = DataLoader(
    train_dataset,
    batch_size=train_config["batch_size"],
    shuffle=train_config["shuffle"],
    num_workers=train_config["num_workers"],
    pin_memory=train_config["pin_memory"],
    prefetch_factor=train_config["prefetch_factor"],
)

# keep validation and test sequential across the two datasets
val_loader = itertools.chain(val_loader1, val_loader2)
test_loader = itertools.chain(test_loader1, test_loader2)



In [4]:
batch1 = next(iter(train_loader))
batch1[4].shape

torch.Size([130, 3, 258, 3])